In [2]:
import os
os.chdir('..')

In [2]:
!pip install bs4

In [3]:
import pandas as pd
from bs4 import BeautifulSoup as BS
import re

In [4]:
data = pd.read_csv('./data_collecting/conversations.csv')


In [35]:
data.text_req[19]

'Здравствуйте! Вчера сделал заявку через сайт на поставку Терморегулятора \r\nРТ-1200А с доставкой в г. Челябинск. Однако, обратной связи до сих пор нет.\r\n\r\n-- \r\nАлександр Мешков\r\n\r\n'

In [5]:
print(data.html_res[data['html_res'].notna()].shape[0])
data.html_req[data['html_req'].notna()].shape[0]


22839


21921

In [6]:
def remove_html_tags(html_text):
    if pd.notna(html_text):
        html = BS(html_text, 'html.parser')
        return html.get_text(separator=' ')
    return pd.NA
    
def clean_text(text):
    if pd.notna(text):
        text = re.sub(r'[\r\n\t]+', ' ', text)
        text = re.sub(r'[ ]{2,}', ' ', text)
        text = text.strip()
        return text
    return pd.NA


In [8]:
cleaned_html_req = data.html_req.apply(remove_html_tags).apply(clean_text)
cleaned_html_req

KeyboardInterrupt: 

In [8]:
cleaned_html_req[22839]

'Добрый день! \xa0 Просим направить КП на поставку TO2-1x PPM Oxygen Sensor -аналог Teledyne B2C - 4 шт. В КП просим указать наличие/сроки поставки и условия оплаты. \xa0 Карточка компании во вложении. \xa0 С уважением, \xa0 Оксана Олейник | Oksana Oleynik Ведущий специалист по логистике | Key Logistics Specialist 8\xa0800\xa0250 8\xa0052 | +7 (985) 893 39 77 \xa0 o . oleynik @ ngco . pro | www . ngco . pro'

In [ ]:
cleaned_html_res = data.html_res.apply(remove_html_tags).apply(clean_text)
cleaned_html_res

0        У нас с вами обмен по ЭДО, выгруженный докумен...
1        Добрый день, Валентина Леонидовна!   Подписанн...
2        Альбина, доброго дня. Счет с указанием сроков ...
3        Добрый день! нет не выиграли   --  С уважением...
4        Акт сверки во вложении   --  С уважением, Гром...
                               ...                        
22836    Здравствуйте! Счет на оплату во вложении.   --...
22837    Олег, продукция по Вашему заказу отправлена ТК...
22838    Здравствуйте!   Высылаю счет.   Он действует 2...
22839    Здравствуйте, Оксана!   Высылаю счет, согласно...
22840    вчера приборы выехали. Вот номер отслеживания:...
Name: html_res, Length: 22841, dtype: object

In [9]:
att = pd.read_csv('./data_collecting/answered_attachments.csv')

In [10]:
uids_with_attachments = att['uid'][att['dispose'] == 'attachment'].drop_duplicates()
uids_list = uids_with_attachments.values
uids_list

array([    1,     2,     5, ..., 33023, 33025, 33028], shape=(12177,))

In [11]:
data_wout_att = data[~data['uid_req'].isin(uids_list)]
data_req = data_wout_att.html_req.fillna(data_wout_att.text_req)
data_req

0        <html xmlns:v="urn:schemas-microsoft-com:vml" ...
2        <html xmlns:v="urn:schemas-microsoft-com:vml" ...
3        <html xmlns:v="urn:schemas-microsoft-com:vml" ...
6        \n<HTML><BODY><div><div>Добрый день!</div><div...
7        <html xmlns:v="urn:schemas-microsoft-com:vml" ...
                               ...                        
22834    <!DOCTYPE html>\r\n<html>\r\n    <head>\r\n   ...
22836    <div>Здравствуйте.</div><div>Сделайте счет на ...
22837    <html><head><meta http-equiv="content-type" co...
22838    <div>Только количество 1шт!</div><div><br /></...
22840    \n<HTML><BODY><div><div>Добрый день!</div><div...
Name: html_req, Length: 13239, dtype: object

In [12]:
data_wout_att = data[~data['uid_req'].isin(uids_list)]
data_res = data_wout_att.html_res.fillna(data_wout_att.text_res)
data_res

0        <div>У нас с вами обмен по ЭДО, выгруженный до...
2        <div>Альбина, доброго дня.</div><div>Счет с ук...
3        <div>Добрый день!</div><div>нет не выиграли</d...
6        <div><div>Добрый день!</div><div>Продукция по ...
7        <div>Мария Григорьевна, здравствуйте!</div><di...
                               ...                        
22834    <div>Здравствуйте, Елена!</div><div>Счет на оп...
22836    <div>Здравствуйте!</div><div>Счет на оплату во...
22837    <div>Олег,</div><div><div style="background-co...
22838    <div>Здравствуйте!</div><div> </div><div>Высыл...
22840    <div>вчера приборы выехали. Вот номер отслежив...
Name: html_res, Length: 13239, dtype: object

In [ ]:
data_req = data_req.apply(remove_html_tags).apply(clean_text)


In [18]:
data_res = data_res.apply(remove_html_tags).apply(clean_text)
data_res

0        У нас с вами обмен по ЭДО, выгруженный докумен...
2        Альбина, доброго дня. Счет с указанием сроков ...
3        Добрый день! нет не выиграли   --  С уважением...
6        Добрый день! Продукция по Вашему заказу отправ...
7        Мария Григорьевна, здравствуйте!   Тут ситуаци...
                               ...                        
22834    Здравствуйте, Елена! Счет на оплату во вложени...
22836    Здравствуйте! Счет на оплату во вложении.   --...
22837    Олег, продукция по Вашему заказу отправлена ТК...
22838    Здравствуйте!   Высылаю счет.   Он действует 2...
22840    вчера приборы выехали. Вот номер отслеживания:...
Name: html_res, Length: 13239, dtype: object

In [19]:
convers_text = pd.concat([data_wout_att.subject_req, data_req, data_res], axis=1)


In [20]:
convers_text.columns = ['subj_req','req', 'res']

In [21]:
convers_wout_re = convers_text[~convers_text['subj_req'].str.contains('RE\[[0-9]+\]:|RE:|FW:|FWD:', case=False, na=False)]


In [22]:
convers_wout_re.to_csv('./data_collecting/convers_cleaned.csv')
convers_wout_re['subj_req'].to_csv('./data_collecting/convers_re.csv')